# Catching fraudsters with statistics

**Question.** Can we recover each detective tool’s accusation from the ledger alone? Each section derives the calculation in pandas, NumPy, SciPy, or scikit-learn, draws the evidence, and checks the implicated IDs against the game’s `run_tool`. The named level supplies the relevant fraud pattern; summary remains descriptive.

In [ ]:
import sys
sys.path.insert(0, "../backend")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare, chi2_contingency, binomtest, ttest_1samp
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import IsolationForest
from auditor.generator.base import (CATEGORY_PROFILES, HONEST_SIGMA, HONEST_Z_CLIP, CEO_MULTIPLIER, make_suspects, make_honest_transactions)
from auditor.generator.cases import build_case, build_honest_case, CASE_START
from auditor.generator.schemes import inject, SCHEME_CATCHERS
from auditor.generator.red_herrings import add_herring
from auditor.tools.registry import run_tool, TOOLS
sns.set_theme(style="darkgrid", context="notebook", palette=["#F5A524", "#4CC9F0", "#E5484D", "#3DD68C"])
plt.rcParams.update({"figure.facecolor":"#0E1117", "axes.facecolor":"#161B24", "axes.labelcolor":"#E8E6E1", "text.color":"#E8E6E1", "xtick.color":"#E8E6E1", "ytick.color":"#E8E6E1", "grid.color":"#394252", "savefig.facecolor":"#0E1117"})
AMBER, CYAN, RED = "#F5A524", "#4CC9F0", "#E5484D"

In [ ]:
cases={level:build_case(level,seed=7) for level in range(1,7)}
def check(tool, level, hand, params=None):
    analysis,_,_=run_tool(tool,cases[level].df,params or {})
    assert sorted(hand)==analysis.implicates,(tool,hand,analysis.implicates)
    print(f"{tool} · level {level}: hand {sorted(hand)} = game {analysis.implicates}; culprit {cases[level].truth.fraudster_id}")
    return analysis

## 1. Summary — descriptive statistics

**Question.** Who spent the most? `describe()` gives count, centre, spread, and extrema; totals are a separate sum. This tool deliberately accuses nobody because high spending can be authorized.

In [ ]:
df=cases[1].df
summary=df.groupby("employee_id").amount.describe()[["count","mean","std","min","50%","max"]].rename(columns={"50%":"median"}).round(2)
summary["total"]=df.groupby("employee_id").amount.sum().round(2)
print(summary)
fig,ax=plt.subplots(figsize=(7,3)); summary.total.plot.barh(ax=ax,color=AMBER); ax.set(title="Total expense by employee",xlabel="INR",ylabel="Employee"); plt.show()
a=check("summary",1,[])
pd.testing.assert_frame_equal(summary.reset_index(),a.table)

**Takeaway.** The top spender is a lead for investigation, not a verdict.

## 2. Groupby — exclusive vendor use

**Question.** Who repeatedly pays a vendor nobody else uses? A vendor crosstab counts claims. The vendor mode uses exclusivity because normal people have favourite vendors. Other groupings use a 2×k chi-square independence test against everyone else, with p < 0.001.

In [ ]:
df=cases[6].df
matrix=pd.crosstab(df.employee_id,df.vendor)
exclusive=matrix.loc[:,(matrix.gt(0).sum(axis=0)==1)].sum(axis=1)
hand=exclusive[exclusive>=5].index.tolist()
fig,ax=plt.subplots(figsize=(11,3.2)); sns.heatmap(matrix,cmap="YlOrBr",ax=ax,cbar_kws={"label":"Claims"}); ax.set(title="Vendor count matrix: look for an exclusive repeated column",xlabel="Vendor",ylabel="Employee"); plt.xticks(rotation=70,ha="right"); plt.tight_layout(); plt.show()
print("Exclusive-vendor claim counts:",exclusive.to_dict())
check("groupby",6,hand,{"by":"vendor"})

**Takeaway.** An exclusive vendor used at least five times is stronger evidence than an ordinary favourite.

## 3. Z-score — comparable claims

**Question.** Which claims differ from the *same person’s same category*? Since category amounts are log-normal, compute `z=(log10(amount)−group mean)/group sample SD`. A claim beyond |z| > 3 implicates its owner. Raw global rupee z-scores would mix categories and the CEO.

In [ ]:
df=cases[1].df
logs=np.log10(df.amount); groups=logs.groupby([df.employee_id,df.category]); z=(logs-groups.transform("mean"))/groups.transform("std")
hand=sorted(df.loc[z.abs()>3,"employee_id"].unique())
fig,ax=plt.subplots(figsize=(8,3)); sns.scatterplot(x=z,y=df.employee_id,hue=z.abs()>3,palette={False:CYAN,True:RED},legend=False,ax=ax,s=22); ax.axvline(3,color=AMBER,ls="--"); ax.axvline(-3,color=AMBER,ls="--"); ax.set(title="Within-person, within-category log z-scores",xlabel="z-score",ylabel="Employee"); plt.show()
check("zscore",1,hand,{"scope":"per_employee","threshold":3.0})

**Takeaway.** This baseline exposes the oversized invoice without treating the CEO’s normal scale as guilt.

## 4. Box plot — Tukey far-out fences

**Question.** Is any claim huge relative to its owner’s usual claim in that category? Subtract each employee–category median on the log scale. For each employee, flag values above `Q3 + 3×(Q3−Q1)`; this is a far-out fence, stricter than the usual 1.5×IQR whisker.

In [ ]:
df=cases[1].df
logs=np.log10(df.amount); relative=logs-logs.groupby([df.employee_id,df.category]).transform("median")
q1=relative.groupby(df.employee_id).transform(lambda x:x.quantile(.25)); q3=relative.groupby(df.employee_id).transform(lambda x:x.quantile(.75))
out=relative>q3+3*(q3-q1); hand=sorted(df.loc[out,"employee_id"].unique())
fig,ax=plt.subplots(figsize=(7,3)); sns.boxplot(x=df.employee_id,y=relative,showfliers=False,color=CYAN,ax=ax); sns.scatterplot(x=df.loc[out,"employee_id"],y=relative[out],color=RED,s=75,ax=ax); ax.set(title="Claim size relative to own category median",xlabel="Employee",ylabel="log₁₀(amount / usual)"); plt.show()
check("iqr_box",1,hand)

**Takeaway.** A far-out point is useful evidence when the baseline is local to that employee.

## 5. Duplicates — receipt twins

**Question.** Did the same person submit the same vendor and amount again within seven days? Join claims to themselves, retain positive day gaps ≤7, and require at least three pairs. The invoice ID is intentionally absent from the join.

In [ ]:
df=cases[2].df
claims=df[["txn_id","employee_id","vendor","amount","date"]].assign(date=pd.to_datetime(df.date))
pairs=claims.merge(claims,on=["employee_id","vendor","amount"],suffixes=("_a","_b"))
gap=(pairs.date_b-pairs.date_a).dt.days; twins=pairs[gap.between(1,7)]
counts=twins.employee_id.value_counts(); hand=sorted(counts[counts>=3].index)
fig,ax=plt.subplots(figsize=(7,3)); counts.reindex(sorted(df.employee_id.unique()),fill_value=0).plot.bar(ax=ax,color=RED); ax.axhline(3,color=AMBER,ls="--"); ax.set(title="Same-vendor, same-amount pairs within seven days",xlabel="Employee",ylabel="Twin pairs"); plt.show()
check("duplicates",2,hand)

**Takeaway.** Reissued IDs do not change the repeated economic transaction.

## 6. Benford — a digit fingerprint

**Question.** Are a suspect’s first digits too far from `P(d)=log10(1+1/d)`? For each employee with at least 50 claims, Pearson’s chi-square statistic compares observed and expected digit counts. The game uses a deliberately strict p < 10⁻⁵ threshold.

In [ ]:
df=cases[3].df
digits=np.floor(df.amount/10**np.floor(np.log10(df.amount))).astype(int)
expected=np.log10(1+1/np.arange(1,10)); digit_counts=pd.crosstab(df.employee_id,digits).reindex(columns=range(1,10),fill_value=0)
pvals={e:chisquare(row.to_numpy(),expected*row.sum()).pvalue for e,row in digit_counts.iterrows() if row.sum()>=50}
hand=sorted(e for e,p in pvals.items() if p<1e-5)
fig,ax=plt.subplots(figsize=(8,3.2)); ax.plot(range(1,10),expected,color=AMBER,ls="--",marker="o",label="Benford")
for e,row in digit_counts.iterrows(): ax.plot(range(1,10),row/row.sum(),alpha=.8,label=e,color=RED if e==cases[3].truth.fraudster_id else None)
ax.set(title="First-digit distribution by employee",xlabel="First digit",ylabel="Share",xticks=range(1,10)); ax.legend(ncol=3); plt.show()
print("Chi-square p-values:",pvals)
check("benford",3,hand)

**Takeaway.** The threshold tests a distribution, not a single suspicious number; small samples stay inconclusive.

## 7. Timeline — weekend rate

**Question.** Who files too many claims when the office is closed? For each employee, estimate the null weekend rate from *everyone else*, then use a one-sided exact binomial test. The rule is p < 0.001.

In [ ]:
df=cases[4].df
dated=df.assign(weekend=pd.to_datetime(df.date).dt.day_of_week>=5)
pvals={e:binomtest(int(rows.weekend.sum()),len(rows),dated.loc[dated.employee_id!=e,"weekend"].mean(),alternative="greater").pvalue for e,rows in dated.groupby("employee_id")}
hand=sorted(e for e,p in pvals.items() if p<.001)
shares=dated.groupby("employee_id").weekend.mean()
fig,ax=plt.subplots(figsize=(7,3)); shares.plot.bar(ax=ax,color=[RED if e in hand else CYAN for e in shares.index]); ax.set(title="Share of claims filed on weekends",xlabel="Employee",ylabel="Weekend share"); plt.show()
print("One-sided binomial p-values:",pvals)
check("timeseries",4,hand,{"freq":"D"})

**Takeaway.** A company-wide weekday bonus cluster does not mimic one employee’s repeated weekend activity.

## 8. Regression — expense against deal size

**Question.** Does one employee consistently spend more than their deals predict? Fit ordinary least squares on `log10(expense)` against `log10(revenue)`, divide residuals by their population SD, then use a one-sided one-sample t-test for mean residual > 0. The game implicates p < 0.001.

In [ ]:
df=cases[5].df
x=np.log10(df[["revenue"]].to_numpy()); y=np.log10(df.amount.to_numpy())
model=LinearRegression().fit(x,y); residual=(y-model.predict(x)); standardized=pd.Series(residual/residual.std(),index=df.index)
pvals={e:ttest_1samp(r,0,alternative="greater").pvalue for e,r in standardized.groupby(df.employee_id)}
hand=sorted(e for e,p in pvals.items() if p<.001)
fig,ax=plt.subplots(figsize=(7,3.5)); sns.scatterplot(x=x[:,0],y=y,hue=df.employee_id,palette="tab10",alpha=.45,s=18,ax=ax); xs=np.linspace(x.min(),x.max(),100); ax.plot(xs,model.predict(xs.reshape(-1,1)),color=AMBER,lw=2,label="OLS fit"); ax.set(title="Log expense vs log deal revenue",xlabel="log₁₀(revenue)",ylabel="log₁₀(amount)"); plt.show()
print("Mean residual p-values:",pvals)
check("regression",5,hand)

**Takeaway.** Consistently positive residuals reveal padded expenses even when individual invoices look plausible.

## 9. Isolation Forest — unusual combinations

**Question.** Who owns too many unusual claims? Features are log size relative to the employee–category median, hour, and vendor popularity. A 200-tree IsolationForest with 2% contamination and seed 0 labels unusual rows. Implicate an employee only with at least six flagged rows and a flagged share above three times the median employee share.

In [ ]:
df=cases[6].df
log_amount=np.log10(df.amount)
features=pd.DataFrame({"unusual_size":log_amount-log_amount.groupby([df.employee_id,df.category]).transform("median"),"hour":df.hour,"vendor_popularity":df.vendor.map(df.vendor.value_counts(normalize=True))})
forest=IsolationForest(n_estimators=200,contamination=.02,random_state=0)
flagged=forest.fit_predict(features)==-1
share=pd.Series(flagged,index=df.index).groupby(df.employee_id).mean(); count=pd.Series(flagged,index=df.index).groupby(df.employee_id).sum()
hand=sorted(share[(count>=6)&(share>3*share.median())].index)
fig,ax=plt.subplots(figsize=(7,3.5)); ax.scatter(df.hour[~flagged],df.amount[~flagged],s=12,alpha=.3,color=CYAN,label="Ordinary"); ax.scatter(df.hour[flagged],df.amount[flagged],s=50,color=RED,label="Isolated"); ax.axhline(50000,color=AMBER,ls="--",label="INR 50k limit"); ax.set(xlabel="Hour",ylabel="Amount (INR)",yscale="log",title="Forest flags unusual size, time, and vendor combinations"); ax.legend(); plt.show()
print(pd.DataFrame({"flagged_count":count,"flagged_share":share}))
check("isoforest",6,hand)

**Takeaway.** Isolation is a multivariate clue. The share and count rules keep a handful of odd honest claims from naming a culprit.

## Is the game fair?

**Question.** Across seeds 0–19, how often does a matching-level tool identify the actual fraudster with at most two implicated employees? The heatmap includes all nine tools; pale cells for non-catching tools are descriptive rather than contract failures. The table below runs every tool on twenty honest companies with both red herrings and reports the fraction with *no* implicated employee.

The stated contract is ≥95% for each catching tool and ≥90% no-accusation rate on honest ledgers. We compute the rates rather than presuming the implementation meets them.

In [ ]:
tools=["summary","groupby","zscore","iqr_box","duplicates","benford","timeseries","regression","isoforest"]
params={"groupby":{"by":"vendor"},"zscore":{"scope":"per_employee"},"timeseries":{"freq":"D"}}
# The groupby catcher uses vendor exclusivity; other params are their documented defaults.
rates=pd.DataFrame(index=range(1,7),columns=tools,dtype=float)
for level in rates.index:
    hits={t:0 for t in tools}
    for seed in range(20):
        case=build_case(level,seed)
        for tool in tools:
            result=TOOLS[tool].analyze(case.df,**params.get(tool,{}))
            hits[tool]+=case.truth.fraudster_id in result.implicates and len(result.implicates)<=2
    rates.loc[level]=pd.Series(hits)/20
fig,ax=plt.subplots(figsize=(11,4)); sns.heatmap(rates,annot=True,fmt=".0%",cmap="YlOrBr",vmin=0,vmax=1,ax=ax,cbar_kws={"label":"Detection rate"}); ax.set(title="Fraudster detected with at most two implicates · 20 seeds per level",xlabel="Tool",ylabel="Level"); plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()
for level,scheme in enumerate(["big_invoice","duplicates","fabricated","weekend","inflated_ratio","under_limit"],1):
    for tool in SCHEME_CATCHERS[scheme]:
        print(f"L{level} {tool}: {rates.loc[level,tool]:.0%} (target ≥95%)")

In [ ]:
clear={t:0 for t in tools}
for seed in range(20):
    honest_case=build_honest_case(seed)
    for tool in tools:
        result=TOOLS[tool].analyze(honest_case.df,**params.get(tool,{}))
        clear[tool]+=not result.implicates
false_positive=pd.DataFrame({"tool":tools,"no_accusation_rate":[clear[t]/20 for t in tools]})
false_positive["false_positive_rate"]=1-false_positive.no_accusation_rate
false_positive["meets_90pct_target"]=false_positive.no_accusation_rate>=.90
print(false_positive)
print("All catching rates ≥95%:",all(rates.loc[level,tool]>=.95 for level,scheme in enumerate(["big_invoice","duplicates","fabricated","weekend","inflated_ratio","under_limit"],1) for tool in SCHEME_CATCHERS[scheme]))
print("All honest no-accusation rates ≥90%:",false_positive.meets_90pct_target.all())

**Takeaway.** The matrix measures detection and specificity separately. A strong game tool needs both: catch its intended pattern and avoid naming innocent employees, including the CEO and bonus recipients.